# 003: CodeExec — Strategy Selection + Code Execution

This notebook walks through every feature added in spec 003. By the end you'll understand:

1. **Strategy ABC** — How execution strategies are defined and registered
2. **ReactStrategy** — The thin wrapper around `react_loop`
3. **ExecuteTool** — Sandboxed Python subprocess execution
4. **CodeExecStrategy** — Prompt augmentation for code-first problem solving
5. **Model-Based Selection** — How the model picks a strategy when multiple are allowed
6. **Integration** — Full end-to-end flow through `run()`

Everything runs with a `MockModel` — no API keys needed.

In [1]:
# Setup: add project paths so imports work from the notebook
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
sys.path.insert(0, os.path.abspath('../tests'))

---

## 1. Strategy ABC

Before 003, `react_loop` was a bare function. Now there's a formal interface:

```python
class Strategy(ABC):
    @property
    @abstractmethod
    def name(self) -> str: ...

    @property
    @abstractmethod
    def description(self) -> str: ...

    @abstractmethod
    async def __call__(self, model, state, sandbox, max_turns) -> LoopResult: ...
```

Every strategy has a `name` (for selection), a `description` (shown to the model during selection), and a `__call__` (the actual execution). Let's see it in action.

In [2]:
from arcrun.strategies import Strategy

# Strategy is abstract — you can't instantiate it directly
try:
    s = Strategy()
except TypeError as e:
    print(f"Can't instantiate Strategy directly: {e}")

Can't instantiate Strategy directly: Can't instantiate abstract class Strategy without an implementation for abstract methods '__call__', 'description', 'name'


In [3]:
# Both built-in strategies are proper subclasses
from arcrun.strategies.react import ReactStrategy
from arcrun.strategies.code import CodeExecStrategy

react = ReactStrategy()
code = CodeExecStrategy()

print(f"ReactStrategy")
print(f"  name:        {react.name}")
print(f"  description: {react.description}")
print(f"  is Strategy: {isinstance(react, Strategy)}")
print()
print(f"CodeExecStrategy")
print(f"  name:        {code.name}")
print(f"  description: {code.description}")
print(f"  is Strategy: {isinstance(code, Strategy)}")

ReactStrategy
  name:        react
  description: Iterative tool-calling loop. Reasons about the task, calls tools, observes results, and repeats until complete. Best for multi-step problems requiring tool interaction.
  is Strategy: True

CodeExecStrategy
  name:        code
  description: Write and execute Python code to solve tasks. Best for computation, data processing, and problems where code is more effective than predefined tool calls.
  is Strategy: True


In [4]:
# Strategies are registered in a global dict, loaded lazily
from arcrun.strategies import STRATEGIES, _load_strategies

_load_strategies()

print("Registered strategies:")
for name, strat in STRATEGIES.items():
    print(f"  {name}: {type(strat).__name__}")

Registered strategies:
  react: ReactStrategy
  code: CodeExecStrategy


### Key design decision

`react_loop` stays as a standalone function. Both `ReactStrategy` and `CodeExecStrategy` call it:

```
ReactStrategy.__call__  ──►  react_loop()
CodeExecStrategy.__call__  ──►  react_loop()  (after augmenting prompt)
```

`react_loop` is the reusable engine. Strategies are the interface layer.

---

## 2. ReactStrategy — Wrapper in Action

Let's run ReactStrategy with a mock model to see the full loop lifecycle. We'll use the same `MockModel` from the test suite.

In [5]:
from conftest import MockModel, LLMResponse, ToolCall, Message
from arcrun.events import EventBus
from arcrun.registry import ToolRegistry
from arcrun.sandbox import Sandbox
from arcrun.state import RunState
from arcrun.types import Tool

# A simple echo tool
async def echo_execute(params, ctx):
    return f"echo: {params.get('input', '')}"

echo_tool = Tool(
    name="echo",
    description="Echo input back",
    input_schema={"type": "object", "properties": {"input": {"type": "string"}}},
    execute=echo_execute,
)

In [6]:
# Scenario: model calls echo tool, then responds with final answer
model = MockModel([
    # Turn 1: model decides to call the echo tool
    LLMResponse(
        content="Let me echo that for you.",
        tool_calls=[ToolCall(id="tc1", name="echo", arguments={"input": "hello world"})],
        stop_reason="tool_use",
    ),
    # Turn 2: model sees the tool result and gives final answer
    LLMResponse(
        content="The echo returned: hello world",
        stop_reason="end_turn",
    ),
])

bus = EventBus(run_id="demo-react")
registry = ToolRegistry(tools=[echo_tool], event_bus=bus)
state = RunState(
    messages=[
        Message(role="system", content="You are a helpful assistant."),
        Message(role="user", content="Echo hello world"),
    ],
    registry=registry,
    event_bus=bus,
    run_id="demo-react",
)
sandbox = Sandbox(config=None, event_bus=bus)

react = ReactStrategy()
result = await react(model, state, sandbox, max_turns=5)

print(f"content:         {result.content}")
print(f"turns:           {result.turns}")
print(f"tool_calls_made: {result.tool_calls_made}")
print(f"strategy_used:   {result.strategy_used}")

content:         The echo returned: hello world
turns:           2
tool_calls_made: 1
strategy_used:   react


In [7]:
# Every action emitted an event — the full audit trail
print(f"Total events: {len(result.events)}\n")
for event in result.events:
    print(f"  {event.type:20s}  {event.data}")

Total events: 10

  loop.start            {'task': 'Echo hello world', 'tool_names': ['echo'], 'strategy': 'react'}
  turn.start            {'turn_number': 1}
  llm.call              {'model': 'MockModel', 'stop_reason': 'tool_use', 'tokens': {'input': 10, 'output': 5, 'total': 15}, 'latency_ms': 0.00095367431640625, 'cost_usd': 0.001}
  tool.start            {'name': 'echo', 'arguments': {'input': 'hello world'}}
  tool.end              {'name': 'echo', 'result_length': 17, 'duration_ms': 0.0019073486328125}
  turn.end              {'turn_number': 1}
  turn.start            {'turn_number': 2}
  llm.call              {'model': 'MockModel', 'stop_reason': 'end_turn', 'tokens': {'input': 20, 'output': 10, 'total': 30}, 'latency_ms': 0.00095367431640625, 'cost_usd': 0.001}
  turn.end              {'turn_number': 2}
  loop.complete         {'content': 'The echo returned: hello world', 'turns': 2, 'tool_calls': 1, 'tokens': {'input': 20, 'output': 10, 'total': 30}, 'cost': 0.002}


Notice the event sequence:
1. `loop.start` — loop begins
2. `turn.start` → `llm.call` → `tool.start` → `tool.end` → `turn.end` — first turn (tool call)
3. `turn.start` → `llm.call` → `turn.end` — second turn (final answer)
4. `loop.complete` — done

Every action is auditable. Always.

---

## 3. ExecuteTool — Sandboxed Python Execution

`make_execute_tool()` creates a tool that runs Python code in a subprocess. Key properties:

- **Process isolation**: `start_new_session=True` creates a new process group
- **Minimal environment**: Only PATH, HOME=/tmp, LANG — no env variable leaks
- **Temp directory**: Code runs in a fresh `TemporaryDirectory`
- **Two-phase timeout**: SIGTERM → 5s grace period → SIGKILL
- **Output truncation**: Caps stdout/stderr at `max_output_bytes`
- **Structured result**: Returns JSON with `{stdout, stderr, exit_code, duration_ms}`

In [8]:
from arcrun.builtins import make_execute_tool

exec_tool = make_execute_tool(timeout_seconds=10)

print(f"name:            {exec_tool.name}")
print(f"description:     {exec_tool.description}")
print(f"timeout_seconds: {exec_tool.timeout_seconds}  (None = managed internally)")
print(f"input_schema:    {exec_tool.input_schema}")

name:            execute_python
description:     Execute Python code in a sandboxed subprocess. Returns stdout, stderr, exit_code, and duration.
timeout_seconds: None  (None = managed internally)
input_schema:    {'type': 'object', 'properties': {'code': {'type': 'string', 'description': 'Python code to execute'}}, 'required': ['code']}


In [9]:
import json
import asyncio
from arcrun.types import ToolContext

# Create a minimal ToolContext for direct execution
ctx = ToolContext(
    run_id="demo",
    tool_call_id="tc-demo",
    turn_number=1,
    event_bus=None,
    cancelled=asyncio.Event(),
)

# Run: simple computation
raw = await exec_tool.execute({"code": "print(2 + 2)"}, ctx)
result = json.loads(raw)

print("Simple computation:")
print(f"  stdout:      {result['stdout']!r}")
print(f"  stderr:      {result['stderr']!r}")
print(f"  exit_code:   {result['exit_code']}")
print(f"  duration_ms: {result['duration_ms']}")

Simple computation:
  stdout:      '4\n'
  stderr:      ''
  exit_code:   0
  duration_ms: 25.4


In [10]:
# Run: code with an error
raw = await exec_tool.execute({"code": "raise ValueError('oops')"}, ctx)
result = json.loads(raw)

print("Error case:")
print(f"  exit_code: {result['exit_code']}")
print(f"  stderr:    {result['stderr'].strip()[-80:]}...")  # last 80 chars

Error case:
  exit_code: 1
  stderr:    um/script.py", line 1, in <module>
    raise ValueError('oops')
ValueError: oops...


In [11]:
# Run: multi-line computation
code = """
import math
primes = [n for n in range(2, 50) if all(n % i != 0 for i in range(2, int(math.sqrt(n)) + 1))]
print(f"Primes under 50: {primes}")
print(f"Count: {len(primes)}")
"""
raw = await exec_tool.execute({"code": code}, ctx)
result = json.loads(raw)

print("Multi-line computation:")
print(result['stdout'])

Multi-line computation:
Primes under 50: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
Count: 15



In [12]:
# Verify env isolation — HOME is /tmp, not your actual home
raw = await exec_tool.execute({"code": "import os; print(os.environ.get('HOME', 'NOT SET'))"}, ctx)
result = json.loads(raw)

print(f"HOME inside subprocess: {result['stdout'].strip()}")
print(f"(Your actual HOME is not leaked)")

HOME inside subprocess: /tmp
(Your actual HOME is not leaked)


In [13]:
# Verify temp dir isolation — cwd is a fresh temp directory
raw = await exec_tool.execute({"code": "import os; print(os.getcwd())"}, ctx)
result = json.loads(raw)

print(f"CWD inside subprocess: {result['stdout'].strip()}")
print(f"(Temp dir — cleaned up after execution)")

CWD inside subprocess: /private/var/folders/c9/55bs__4x0_n_gns0yrptxv5m0000gn/T/tmpraug_0nw
(Temp dir — cleaned up after execution)


In [14]:
# Extra env: add custom environment variables
custom_tool = make_execute_tool(extra_env={"MY_VAR": "secret_value"})
raw = await custom_tool.execute({"code": "import os; print(os.environ.get('MY_VAR'))"}, ctx)
result = json.loads(raw)

print(f"Custom env var: {result['stdout'].strip()}")

Custom env var: secret_value

### Timeout behavior

When code exceeds the timeout, ExecuteTool uses two-phase shutdown:
1. **SIGTERM** sent to the entire process group
2. Wait 5 seconds (grace period)
3. **SIGKILL** if still running

This ensures child processes are cleaned up even if the main script ignores SIGTERM.

In [15]:
# Timeout: 1-second timeout on a 10-second sleep
fast_tool = make_execute_tool(timeout_seconds=1)
raw = await fast_tool.execute({"code": "import time; time.sleep(10); print('done')"}, ctx)
result = json.loads(raw)

print(f"Timed out:")
print(f"  exit_code: {result['exit_code']}")
print(f"  stderr:    {result['stderr']}")
print(f"  stdout:    {result['stdout']!r}  (empty — never reached print)")

Timed out:
  exit_code: -1
  stderr:    Error: execution timed out
  stdout:    ''  (empty — never reached print)


---

## 4. CodeExecStrategy — The LLM Writes Code, the Engine Runs It

Here's how CodeExec works:

1. **System prompt is augmented** with instructions telling the LLM it has an `execute_python` tool
2. The LLM **writes a Python script** and returns it as a tool call
3. **ExecuteTool runs that script** in a sandboxed subprocess
4. The structured result (`stdout`, `stderr`, `exit_code`, `duration_ms`) is **fed back to the LLM**
5. The LLM **interprets the output** and either writes more code or gives a final answer

The loop itself is identical to ReAct — the only difference is the system prompt tells the LLM to solve problems by writing and running code.

Let's see it live, then audit every step.

In [16]:
from arcrun.strategies.code import CodeExecStrategy, _DEFAULT_PREFIX

print("Default code execution prefix:")
print("─" * 60)
print(_DEFAULT_PREFIX)
print("─" * 60)

Default code execution prefix:
────────────────────────────────────────────────────────────
You have access to a Python execution tool (execute_python). Write executable Python code to solve tasks.

GUIDELINES:
- Write focused scripts (20-50 lines) solving one sub-problem at a time
- You will receive {stdout, stderr, exit_code, duration_ms} after each execution
- Each execution is stateless - variables do NOT persist between calls
- If code fails, examine the error and fix your approach
- After 3 failures on the same approach, try a fundamentally different method
- Use code for: computation, data processing, logic, file operations
- Use other tools for: external APIs, user confirmation, security-sensitive ops

────────────────────────────────────────────────────────────


In [17]:
# Live demo: LLM writes a Python script to solve a math problem
# ─────────────────────────────────────────────────────────────
# The MockModel simulates what a real LLM does:
#   Turn 1: Write Python code in an execute_python tool call
#   Turn 2: Read stdout/stderr, give final answer

exec_tool = make_execute_tool(timeout_seconds=5)

model = MockModel([
    # Turn 1: LLM writes a Python script
    LLMResponse(
        content="I'll write code to compute the sum.",
        tool_calls=[ToolCall(
            id="tc1",
            name="execute_python",
            arguments={"code": "total = sum(range(1, 101))\nprint(f'The sum of 1..100 is {total}')"},
        )],
        stop_reason="tool_use",
    ),
    # Turn 2: LLM reads the execution result and responds
    LLMResponse(
        content="The sum of numbers 1 through 100 is 5,050.",
        stop_reason="end_turn",
    ),
])

bus = EventBus(run_id="demo-code")
registry = ToolRegistry(tools=[exec_tool], event_bus=bus)
state = RunState(
    messages=[
        Message(role="system", content="You are a helpful assistant."),
        Message(role="user", content="What is the sum of all numbers from 1 to 100?"),
    ],
    registry=registry,
    event_bus=bus,
    run_id="demo-code",
    strategy_name="code",
)
sandbox = Sandbox(config=None, event_bus=bus)

code_strat = CodeExecStrategy()
result = await code_strat(model, state, sandbox, max_turns=5)

print(f"content:         {result.content}")
print(f"strategy_used:   {result.strategy_used}")
print(f"turns:           {result.turns}")
print(f"tool_calls_made: {result.tool_calls_made}")

content:         The sum of numbers 1 through 100 is 5,050.
strategy_used:   code
turns:           2
tool_calls_made: 1


In [18]:
# The system message was augmented — this is what the LLM actually saw
system_msg = state.messages[0].content
print("Augmented system prompt sent to the LLM:")
print("─" * 60)
print(system_msg[:350])
print("─" * 60)
print(f"\nTotal length: {len(system_msg)} chars")
print(f"Starts with code guidelines: {system_msg.startswith('You have access to a Python')}")
print(f"Original prompt preserved at end: {'You are a helpful assistant.' in system_msg}")

Augmented system prompt sent to the LLM:
────────────────────────────────────────────────────────────
You have access to a Python execution tool (execute_python). Write executable Python code to solve tasks.

GUIDELINES:
- Write focused scripts (20-50 lines) solving one sub-problem at a time
- You will receive {stdout, stderr, exit_code, duration_ms} after each execution
- Each execution is stateless - variables do NOT persist between calls
- If co
────────────────────────────────────────────────────────────

Total length: 656 chars
Starts with code guidelines: True
Original prompt preserved at end: True


In [19]:
# The code.prompt.augmented event was emitted
augmented_events = [e for e in result.events if e.type == "code.prompt.augmented"]
print(f"code.prompt.augmented event: {augmented_events[0].data}")

code.prompt.augmented event: {'original_length': 28, 'augmented_length': 656}


### Auditing the Full Flow

This is the observability payoff. We can trace through **every step** of the interaction:

1. What messages were sent to the LLM?
2. What did the LLM return? (the tool call containing the Python script)
3. What did ExecuteTool actually run, and what came back?
4. What was fed back to the LLM as the tool result?
5. How did the LLM interpret the output?

The `MockModel` records every `invoke()` call, and the `EventBus` captures every event.

In [20]:
import json

# ── STEP 1: What messages did the LLM receive? ──────────────────────
print("=" * 70)
print("STEP 1: Messages sent to the LLM")
print("=" * 70)
first_call = model.invoke_calls[0]
for msg in first_call["messages"]:
    content = msg.content if isinstance(msg.content, str) else "[structured]"
    if len(content) > 100:
        content = content[:100] + "..."
    print(f"  [{msg.role:8s}] {content}")
print(f"  Tools:    {[t.name for t in first_call['tools']]}")

# ── STEP 2: What did the LLM return? (the tool call with code) ──────
print(f"\n{'=' * 70}")
print("STEP 2: LLM returned a tool call containing this Python script")
print("=" * 70)
tool_start = [e for e in result.events if e.type == "tool.start"][0]
script = tool_start.data["arguments"]["code"]
print()
print("  ┌─ execute_python ────────────────────────────────")
for line in script.split("\n"):
    print(f"  │  {line}")
print("  └────────────────────────────────────────────────")

# ── STEP 3: What did ExecuteTool produce? ────────────────────────────
print(f"\n{'=' * 70}")
print("STEP 3: ExecuteTool ran it in a sandboxed subprocess")
print("=" * 70)
tool_end = [e for e in result.events if e.type == "tool.end"][0]
print(f"  duration: {tool_end.data['duration_ms']:.1f}ms")

# The tool result is in the messages the model saw on its 2nd invocation
second_call = model.invoke_calls[1]
for msg in second_call["messages"]:
    if msg.role == "tool":
        for block in msg.content:
            parsed = json.loads(block.content)
            print(f"\n  Structured result fed back to the LLM:")
            print(f"    stdout:      {parsed['stdout']!r}")
            print(f"    stderr:      {parsed['stderr']!r}")
            print(f"    exit_code:   {parsed['exit_code']}")
            print(f"    duration_ms: {parsed['duration_ms']}")

# ── STEP 4: How did the LLM interpret it? ────────────────────────────
print(f"\n{'=' * 70}")
print("STEP 4: LLM interpreted the result and gave final answer")
print("=" * 70)
print(f'\n  "{result.content}"')

# ── Full event timeline ──────────────────────────────────────────────
print(f"\n{'=' * 70}")
print("Complete event timeline")
print("=" * 70)
for e in result.events:
    extras = ", ".join(f"{k}={v!r}" for k, v in e.data.items() if k not in ("tokens",))
    print(f"  {e.type:28s} {extras}")

STEP 1: Messages sent to the LLM
  [system  ] You have access to a Python execution tool (execute_python). Write executable Python code to solve t...
  [user    ] What is the sum of all numbers from 1 to 100?
  [assistant] [structured]
  [tool    ] [structured]
  [assistant] [structured]
  Tools:    ['execute_python']

STEP 2: LLM returned a tool call containing this Python script

  ┌─ execute_python ────────────────────────────────
  │  total = sum(range(1, 101))
  │  print(f'The sum of 1..100 is {total}')
  └────────────────────────────────────────────────

STEP 3: ExecuteTool ran it in a sandboxed subprocess
  duration: 28.7ms

  Structured result fed back to the LLM:
    stdout:      'The sum of 1..100 is 5050\n'
    stderr:      ''
    exit_code:   0
    duration_ms: 28.3

STEP 4: LLM interpreted the result and gave final answer

  "The sum of numbers 1 through 100 is 5,050."

Complete event timeline
  code.prompt.augmented        original_length=28, augmented_length=656
  loop.s

In [21]:
# Custom prefix: override the default guidelines
custom = CodeExecStrategy(system_prompt_prefix="Always use list comprehensions.")
print(f"Custom prefix strategy name: {custom.name}")

# Build fresh state to see the custom prefix in action
bus2 = EventBus(run_id="demo-custom")
registry2 = ToolRegistry(tools=[echo_tool], event_bus=bus2)
state2 = RunState(
    messages=[
        Message(role="system", content="Original prompt."),
        Message(role="user", content="Do something."),
    ],
    registry=registry2,
    event_bus=bus2,
    run_id="demo-custom",
)
sandbox2 = Sandbox(config=None, event_bus=bus2)
model2 = MockModel([LLMResponse(content="Done.", stop_reason="end_turn")])

await custom(model2, state2, sandbox2, max_turns=5)
print(f"Augmented system message: {state2.messages[0].content!r}")

Custom prefix strategy name: code
Augmented system message: 'Always use list comprehensions.\nOriginal prompt.'


---

## 5. Model-Based Strategy Selection

When `allowed_strategies` has multiple entries, the model picks which one to use via tool calling. This happens *before* the main loop starts.

The selection logic has three paths:

| Scenario | What happens | LLM calls |
|---|---|---|
| `allowed=None` | Default to "react" | 0 |
| `allowed=["code"]` | Use that strategy directly | 0 |
| `allowed=["react", "code"]` | Model picks via `select_strategy` tool | 1 |

In [22]:
from arcrun.strategies import select_strategy

# Path 1: None → default to "react" (no LLM call)
bus = EventBus(run_id="sel-none")
registry = ToolRegistry(tools=[echo_tool], event_bus=bus)
state = RunState(
    messages=[Message(role="user", content="test")],
    registry=registry,
    event_bus=bus,
    run_id="sel-none",
)

# model is never called for None
chosen = await select_strategy(None, None, state)
print(f"allowed=None → {chosen!r}")

allowed=None → 'react'


In [23]:
# Path 2: single strategy → use directly (no LLM call)
bus = EventBus(run_id="sel-single")
registry = ToolRegistry(tools=[echo_tool], event_bus=bus)
state = RunState(
    messages=[Message(role="user", content="test")],
    registry=registry,
    event_bus=bus,
    run_id="sel-single",
)

chosen = await select_strategy(["code"], None, state)
print(f"allowed=['code'] → {chosen!r}")

allowed=['code'] → 'code'


In [24]:
# Path 3: multiple strategies → model picks via tool call
bus = EventBus(run_id="sel-multi")
registry = ToolRegistry(tools=[echo_tool], event_bus=bus)
state = RunState(
    messages=[Message(role="user", content="Calculate the fibonacci sequence")],
    registry=registry,
    event_bus=bus,
    run_id="sel-multi",
)

# Mock model responds with select_strategy tool call choosing "code"
selection_model = MockModel([
    LLMResponse(
        tool_calls=[ToolCall(
            id="sel1",
            name="select_strategy",
            arguments={"strategy": "code", "reasoning": "computational task benefits from code"},
        )],
        stop_reason="tool_use",
    ),
])

chosen = await select_strategy(["react", "code"], selection_model, state)
print(f"allowed=['react', 'code'] → model chose {chosen!r}")

# Check what events were emitted
for event in bus.events:
    print(f"  {event.type}: {event.data}")

allowed=['react', 'code'] → model chose 'code'
  strategy.selection.start: {'allowed_strategies': ['react', 'code'], 'task': 'Calculate the fibonacci sequence'}
  strategy.selection.complete: {'selected': 'code', 'reasoning': 'computational task benefits from code'}


In [25]:
# Let's see what the model was shown during selection
invoke_call = selection_model.invoke_calls[0]
system_msg = invoke_call["messages"][0].content
user_msg = invoke_call["messages"][1].content

print("System message shown to model for selection:")
print("─" * 60)
print(system_msg)
print("─" * 60)
print(f"\nUser message (the task): {user_msg!r}")
print(f"\nTools provided: {[t.name for t in invoke_call['tools']]}")

System message shown to model for selection:
────────────────────────────────────────────────────────────
Select the best execution strategy for the task below.

Available strategies:
- react: Iterative tool-calling loop. Reasons about the task, calls tools, observes results, and repeats until complete. Best for multi-step problems requiring tool interaction.
- code: Write and execute Python code to solve tasks. Best for computation, data processing, and problems where code is more effective than predefined tool calls.

Available tools: echo

Call select_strategy with your choice.
────────────────────────────────────────────────────────────

User message (the task): 'Calculate the fibonacci sequence'

Tools provided: ['select_strategy']


In [26]:
# Fallback: if model returns invalid strategy, falls back to "react"
bus = EventBus(run_id="sel-fallback")
registry = ToolRegistry(tools=[echo_tool], event_bus=bus)
state = RunState(
    messages=[Message(role="user", content="test")],
    registry=registry,
    event_bus=bus,
    run_id="sel-fallback",
)

# Model returns a strategy name that doesn't exist
bad_model = MockModel([
    LLMResponse(
        tool_calls=[ToolCall(
            id="sel1",
            name="select_strategy",
            arguments={"strategy": "nonexistent"},
        )],
        stop_reason="tool_use",
    ),
])

chosen = await select_strategy(["react", "code"], bad_model, state)
print(f"Invalid model output → fell back to {chosen!r}")

fallback_events = [e for e in bus.events if e.type == "strategy.selection.fallback"]
print(f"Fallback event: {fallback_events[0].data}")

Invalid model output → fell back to 'react'
Fallback event: {'attempted': ['react', 'code'], 'defaulted_to': 'react'}


In [27]:
# Error handling: if the model raises an exception, falls back to "react"
class ExplodingModel:
    async def invoke(self, messages, tools=None):
        raise RuntimeError("API is down")

bus = EventBus(run_id="sel-error")
registry = ToolRegistry(tools=[echo_tool], event_bus=bus)
state = RunState(
    messages=[Message(role="user", content="test")],
    registry=registry,
    event_bus=bus,
    run_id="sel-error",
)

chosen = await select_strategy(["react", "code"], ExplodingModel(), state)
print(f"Model error → fell back to {chosen!r}")

Model error → fell back to 'react'


### Why tool calling for selection?

Using a tool call with `enum` constraint guarantees the model returns a valid strategy name. No string parsing, no regex, no "please output exactly one word". The enum in the JSON schema enforces the valid options:

```python
{"strategy": {"type": "string", "enum": ["react", "code"]}}
```

---

## 6. Integration — Full End-to-End

Now let's see everything work together through `run()`, the main entry point.

In [28]:
from arcrun.loop import run

# Scenario: run() with default strategy (react)
# No allowed_strategies = defaults to "react", no selection call
model = MockModel([
    LLMResponse(
        tool_calls=[ToolCall(id="tc1", name="echo", arguments={"input": "ping"})],
        stop_reason="tool_use",
    ),
    LLMResponse(content="Echo returned: ping", stop_reason="end_turn"),
])

result = await run(
    model=model,
    tools=[echo_tool],
    system_prompt="You are helpful.",
    task="Echo ping",
)

print(f"Strategy: {result.strategy_used}")
print(f"Content:  {result.content}")
print(f"Turns:    {result.turns}")
print(f"Tools:    {result.tool_calls_made}")

Strategy: react
Content:  Echo returned: ping
Turns:    2
Tools:    1


In [29]:
# Scenario: run() with strategy selection (model picks "code")
model = MockModel([
    # First call: strategy selection
    LLMResponse(
        tool_calls=[ToolCall(
            id="sel1",
            name="select_strategy",
            arguments={"strategy": "code"},
        )],
        stop_reason="tool_use",
    ),
    # Second call: model executes with code strategy
    LLMResponse(content="Computed the answer.", stop_reason="end_turn"),
])

result = await run(
    model=model,
    tools=[echo_tool],
    system_prompt="You are helpful.",
    task="Compute 2+2",
    allowed_strategies=["react", "code"],
)

print(f"Strategy: {result.strategy_used}")
print(f"Content:  {result.content}")

# Verify all the selection + execution events
event_types = [e.type for e in result.events]
print(f"\nEvent types in order:")
for et in event_types:
    print(f"  {et}")

Strategy: code
Content:  Computed the answer.

Event types in order:
  strategy.selection.start
  strategy.selection.complete
  strategy.selected
  code.prompt.augmented
  loop.start
  turn.start
  llm.call
  turn.end
  loop.complete


In [30]:
# Scenario: CodeExec end-to-end through run()
# Same flow as section 4, but through the main entry point
exec_tool = make_execute_tool(timeout_seconds=5)

model = MockModel([
    LLMResponse(
        tool_calls=[ToolCall(
            id="tc1",
            name="execute_python",
            arguments={"code": "print(sum(range(101)))"},
        )],
        stop_reason="tool_use",
    ),
    LLMResponse(content="The sum of 0 to 100 is 5050.", stop_reason="end_turn"),
])

result = await run(
    model=model,
    tools=[echo_tool, exec_tool],
    system_prompt="You are helpful.",
    task="What is the sum of numbers 0 to 100?",
    allowed_strategies=["code"],
)

print(f"Strategy: {result.strategy_used}")
print(f"Content:  {result.content}")
print(f"Turns:    {result.turns}")
print(f"Tools:    {result.tool_calls_made}")

# The full audit trail is available here too — see section 4 for the detailed breakdown
tool_events = [e for e in result.events if e.type in ("tool.start", "tool.end")]
for e in tool_events:
    print(f"  {e.type}: {e.data}")

Strategy: code
Content:  The sum of 0 to 100 is 5050.
Turns:    2
Tools:    1
  tool.start: {'name': 'execute_python', 'arguments': {'code': 'print(sum(range(101)))'}}
  tool.end: {'name': 'execute_python', 'result_length': 71, 'duration_ms': 26.103734970092773}


In [31]:
# Scenario: Sandbox denies execute_python
from arcrun.types import SandboxConfig

model = MockModel([
    LLMResponse(
        tool_calls=[ToolCall(
            id="tc1",
            name="execute_python",
            arguments={"code": "print('hack')"},
        )],
        stop_reason="tool_use",
    ),
    LLMResponse(content="I was denied.", stop_reason="end_turn"),
])

result = await run(
    model=model,
    tools=[echo_tool, exec_tool],
    system_prompt="You are helpful.",
    task="Run some code",
    allowed_strategies=["code"],
    sandbox=SandboxConfig(allowed_tools=["echo"]),  # execute_python NOT allowed
)

denied = [e for e in result.events if e.type == "tool.denied"]
print(f"Denied events: {len(denied)}")
print(f"Denied tool:   {denied[0].data['name']}")
print(f"Reason:        {denied[0].data['reason']}")
print(f"Content:       {result.content}")

Denied events: 1
Denied tool:   execute_python
Reason:        execute_python: not in allowed tools
Content:       I was denied.


---

## 7. The Event Trail

Let's do one final run with an `on_event` handler to see the real-time event stream — every action, as it happens.

In [32]:
# Real-time event handler
def live_handler(event):
    # Format: [event_type] key=value ...
    data_str = "  ".join(f"{k}={v!r}" for k, v in event.data.items() if k != "tokens")
    print(f"  [{event.type}] {data_str}")

model = MockModel([
    # Strategy selection
    LLMResponse(
        tool_calls=[ToolCall(id="sel1", name="select_strategy", arguments={"strategy": "code"})],
        stop_reason="tool_use",
    ),
    # Code execution
    LLMResponse(
        content="Computing...",
        tool_calls=[ToolCall(id="tc1", name="execute_python", arguments={"code": "print(7 * 6)"})],
        stop_reason="tool_use",
    ),
    # Final answer
    LLMResponse(content="7 * 6 = 42", stop_reason="end_turn"),
])

print("Live event stream:")
print("═" * 70)

result = await run(
    model=model,
    tools=[echo_tool, exec_tool],
    system_prompt="You are helpful.",
    task="What is 7 * 6?",
    allowed_strategies=["react", "code"],
    on_event=live_handler,
)

print("═" * 70)
print(f"\nFinal: {result.content}")
print(f"Strategy: {result.strategy_used}")
print(f"Turns: {result.turns}, Tools: {result.tool_calls_made}")
print(f"Total events: {len(result.events)}")

Live event stream:
══════════════════════════════════════════════════════════════════════
  [strategy.selection.start] allowed_strategies=['react', 'code']  task='What is 7 * 6?'
  [strategy.selection.complete] selected='code'  reasoning=''
  [strategy.selected] strategy='code'
  [code.prompt.augmented] original_length=16  augmented_length=644
  [loop.start] task='What is 7 * 6?'  tool_names=['echo', 'execute_python']  strategy='react'
  [turn.start] turn_number=1
  [llm.call] model='MockModel'  stop_reason='tool_use'  latency_ms=0.00095367431640625  cost_usd=0.001
  [tool.start] name='execute_python'  arguments={'code': 'print(7 * 6)'}
  [tool.end] name='execute_python'  result_length=69  duration_ms=25.300979614257812
  [turn.end] turn_number=1
  [turn.start] turn_number=2
  [llm.call] model='MockModel'  stop_reason='end_turn'  latency_ms=0.0021457672119140625  cost_usd=0.001
  [turn.end] turn_number=2
  [loop.complete] content='7 * 6 = 42'  turns=2  tool_calls=1  cost=0.002
════════

---

## Summary

Spec 003 added four features to arcrun:

| Feature | What it does | Lines |
|---|---|---|
| **Strategy ABC** | Formal interface for execution strategies (`name`, `description`, `__call__`) | ~28 lines in `strategies/__init__.py` |
| **ExecuteTool** | Sandboxed Python subprocess with process group isolation, two-phase timeout, minimal env | 91 lines in `builtins/execute.py` |
| **CodeExecStrategy** | Augments system prompt with code guidelines, delegates to `react_loop` | 55 lines in `strategies/code.py` |
| **Strategy Selection** | Model picks strategy via tool calling when multiple allowed; enum constraint ensures valid output | ~77 lines in `strategies/__init__.py` |

### Architecture

```
run() → select_strategy() → Strategy.__call__()
                                    │
                    ┌───────────────┴───────────────┐
                    ▼                               ▼
            ReactStrategy                   CodeExecStrategy
            (thin wrapper)              (augment prompt, then...)
                    │                               │
                    └───────────┬───────────────────┘
                                ▼
                          react_loop()
                        (the reusable engine)
```

`react_loop` is the shared engine. Strategies add behavior *around* it (prompt augmentation, context setup) without modifying it. Zero behavior change to existing code.